<a href="https://colab.research.google.com/github/kat-guse/WEME_Localization_Abundance/blob/main/Perch_Individual_Id.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This block processes localized acoustic detections to estimate the number of unique individuals and map their territories. Individual ID estimation used Perch 2 embeddings, t-SNE, and HDBSCAN/KMeans

**Steps:**
1. Match filtered detections back to original clip indices.
2. Extract Perch 2 embeddings from matched .wav clips
3. t-SNE Dimensionality reduction based on Sam Lapp et al. (2025). 3D and 2D
4. Cluster with HDBSCAN (3 (loose) and 7(based on Sam Lapp et al. (2025)) and KMeans (strict)
5. Spatial and Temporal Coherence Validation
6. Visualizing (temporal singing pattern plot, spatial map coloured by cluster)
7. Exporting


In [ ]:
import os
# CUDA before importing tensorflow - Perch2 TF model runs on CPU only
os.environ['CUDA_VISIBLE_DEVICES'] = ''


import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import bioacoustics_model_zoo as bmz
from umap import UMAP
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import hdbscan

# ---------- CONFIGURATION ------------------
CLIP_DIR          = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/minspec_clips'
ORIGINAL_CSV      = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/OKLG-8_Western_Meadowlark_CLEANED.csv'
FILTERED_CSV      = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/weml_confirmed_locations_hull15m_filtered_final.csv'
OUTPUT_DIR        = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/individual_id'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- 1. Match filtered detections back to original clip indices ------
df_filtered = pd.read_csv(FILTERED_CSV)
df_original = pd.read_csv(ORIGINAL_CSV)

df_filtered['timestamp_match'] = pd.to_datetime(
    df_filtered['timestamp'], format='ISO8601'
).dt.tz_localize(None).astype(str).str[:23]

df_original['timestamp_match'] = pd.to_datetime(
    df_original['start_timestamp'], format='mixed'
).dt.tz_localize(None).astype(str).str[:23]

ts_to_idx = dict(zip(df_original['timestamp_match'], df_original.index))
df_filtered['clip_idx'] = df_filtered['timestamp_match'].map(ts_to_idx)

matched   = df_filtered['clip_idx'].notna().sum()
unmatched = df_filtered['clip_idx'].isna().sum()
print(f"  Matched:   {matched} / {len(df_filtered)}")
print(f"  Unmatched: {unmatched}")

df_matched = df_filtered.dropna(subset=['clip_idx']).copy()
df_matched['clip_idx']  = df_matched['clip_idx'].astype(int)
df_matched['clip_path'] = df_matched['clip_idx'].apply(
    lambda i: os.path.join(CLIP_DIR, f"{i}.wav")
)
df_matched['clip_exists'] = df_matched['clip_path'].apply(os.path.exists)
missing = (~df_matched['clip_exists']).sum()
print(f"  Clip files found: {df_matched['clip_exists'].sum()} / {len(df_matched)}")
if missing > 0:
    print(f"  ⚠️  {missing} clip files not found on disk")
    df_matched = df_matched[df_matched['clip_exists']].copy()

print(f"\n  Proceeding with {len(df_matched)} detections\n")

# ------------ 2. Extract Perch 2 embeddings from matched .wav clips ---------
embedding_cache = os.path.join(OUTPUT_DIR, 'embeddings.npy')
index_cache     = os.path.join(OUTPUT_DIR, 'embedding_indices.npy')

if os.path.exists(embedding_cache):
    print("  Loading cached embeddings...")
    embeddings    = np.load(embedding_cache)
    valid_indices = np.load(index_cache)
    df_embedded   = df_matched.iloc[valid_indices].copy().reset_index(drop=True)
    print(f"  Loaded {len(embeddings)} embeddings from cache")
else:
    print("  Loading Perch2 model (CPU mode)...")
    import bioacoustics_model_zoo as bmz
    model = bmz.Perch2(device='cpu')
    print("  Perch2 loaded successfully")

    clip_paths = df_matched['clip_path'].tolist()
    index = pd.MultiIndex.from_tuples(
        [(c, 0.0, 5.0) for c in clip_paths],
        names=['file', 'start_time', 'end_time']
    )
    samples = pd.DataFrame(index=index)

    print(f"  Running inference on {len(clip_paths)} clips...")
    embeddings = model.embed(samples, return_dfs=False)
    print(f"  Raw embedding shape: {embeddings.shape}")

    valid_mask    = ~np.all(embeddings == 0, axis=1) & ~np.any(np.isnan(embeddings), axis=1)
    valid_indices = np.where(valid_mask)[0]
    embeddings    = embeddings[valid_mask]
    df_embedded   = df_matched.iloc[valid_indices].copy().reset_index(drop=True)

    np.save(embedding_cache, embeddings)
    np.save(index_cache, valid_indices)
    print(f"  ✅ Saved {len(embeddings)} embeddings to cache")

print(f"\n  Embeddings ready: {embeddings.shape} ({len(df_embedded)} detections)\n")

# Parse timestamps
df_embedded['timestamp_pdt'] = pd.to_datetime(
    df_embedded['timestamp'], format='ISO8601'
).dt.tz_convert('America/Vancouver').dt.tz_localize(None)

# ------------------- 3. t-SNE Dimensionality reduction (instead of UMAP based on Sam Lapp) ---------------------------
embeddings_scaled = StandardScaler().fit_transform(embeddings)

# t-SNE with 3 dimensions — Sam Lapp found this outperformed UMAP across all metrics
tsne_3d = TSNE(
    n_components=3,
    perplexity=min(30, len(embeddings) // 4),
    random_state=42,
    max_iter=1000,
    metric='cosine'
)
features_3d = tsne_3d.fit_transform(embeddings_scaled)
print(f"  t-SNE 3D complete: {embeddings.shape} → {features_3d.shape}")

# Also reduce to 2D for visualization
tsne_2d = TSNE(
    n_components=2,
    perplexity=min(30, len(embeddings) // 4),
    random_state=42,
    max_iter=1000,
    metric='cosine'
)
features_2d = tsne_2d.fit_transform(embeddings_scaled)
print(f"  t-SNE 2D complete: {embeddings.shape} → {features_2d.shape}\n")

# ------------------ 4. Clustering -----------------------
results = {}

#HUBSCAN - min_cluster_size = 3 (loose)
hdb3 = hdbscan.HDBSCAN(min_cluster_size=3, min_samples=2, metric='euclidean').fit(features_3d)
n_hdb3       = len(set(hdb3.labels_)) - (1 if -1 in hdb3.labels_ else 0)
n_noise_hdb3 = (hdb3.labels_ == -1).sum()
results['hdbscan_mcs3'] = hdb3.labels_
print(f"  HDBSCAN (min_cluster_size=3): {n_hdb3} clusters, {n_noise_hdb3} noise points")

#HUBSCAN - min_cluster_size = 7 (Sam Lapp's value)
hdb7 = hdbscan.HDBSCAN(min_cluster_size=7, min_samples=3, metric='euclidean').fit(features_3d)
n_hdb7       = len(set(hdb7.labels_)) - (1 if -1 in hdb7.labels_ else 0)
n_noise_hdb7 = (hdb7.labels_ == -1).sum()
results['hdbscan_mcs7'] = hdb7.labels_
print(f"  HDBSCAN (min_cluster_size=7): {n_hdb7} clusters, {n_noise_hdb7} noise points")

#KMeans (strict) - best k with silhouette score
print("\n  KMeans silhouette scores:")
k_range    = range(2, min(20, len(embeddings) // 5))
sil_scores = []
for k in k_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10).fit(features_3d)
    sil = silhouette_score(features_3d, km.labels_)
    sil_scores.append(sil)
    print(f"    k={k:>2}: silhouette={sil:.3f}")

best_k   = list(k_range)[np.argmax(sil_scores)]
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(features_3d)
results['kmeans'] = km_final.labels_
print(f"\n  KMeans best k={best_k} (silhouette={max(sil_scores):.3f})")

print(f"\n  Summary:")
print(f"    HDBSCAN (mcs=3): {n_hdb3} individuals + {n_noise_hdb3} unassigned")
print(f"    HDBSCAN (mcs=7): {n_hdb7} individuals + {n_noise_hdb7} unassigned")
print(f"    KMeans:          {best_k} individuals")

# ------------------- 5. Spatial and temporal coherence validation -------------------------
def spatial_coherence(df, labels, label_name):
    """
    For each cluster, compute mean pairwise distance between detections.
    Tight clusters spatially = likely same individual on a territory.
    """
    records = []
    unique_labels = sorted(set(labels))
    for cid in unique_labels:
        if cid == -1:
            continue
        mask = labels == cid
        pts  = df[mask][['x', 'y']].values
        n    = len(pts)
        if n < 2:
            mean_dist = 0.0
        else:
            # mean pairwise distance
            from scipy.spatial.distance import pdist
            mean_dist = pdist(pts).mean()
        records.append({
            'cluster': int(cid),
            'n_detections': int(n),
            'mean_spatial_spread_m': round(mean_dist, 1),
        })
    df_coh = pd.DataFrame(records)
    print(f"\n  {label_name} — Spatial coherence per cluster:")
    print(df_coh.to_string(index=False))
    return df_coh

def temporal_coherence(df, labels, label_name):
    """
    For each cluster, show when it sings (hour distribution).
    Real birds should peak during dawn chorus (04:00-08:00 PDT).
    """
    records = []
    unique_labels = sorted(set(labels))
    for cid in unique_labels:
        if cid == -1:
            continue
        mask  = labels == cid
        times = df[mask]['timestamp_pdt']
        hours = times.dt.hour + times.dt.minute / 60
        records.append({
            'cluster':      int(cid),
            'n_detections': int(mask.sum()),
            'first_song':   times.min().strftime('%H:%M'),
            'last_song':    times.max().strftime('%H:%M'),
            'peak_hour':    f"{hours.mean():.1f}",
            'span_minutes': round((times.max() - times.min()).total_seconds() / 60, 1),
        })
    df_temp = pd.DataFrame(records)
    print(f"\n  {label_name} — Temporal pattern per cluster:")
    print(df_temp.to_string(index=False))
    return df_temp

# Run coherence analysis for all clustering results
spatial_hdb3  = spatial_coherence(df_embedded, hdb3.labels_,  'HDBSCAN mcs=3')
temporal_hdb3 = temporal_coherence(df_embedded, hdb3.labels_, 'HDBSCAN mcs=3')

spatial_hdb7  = spatial_coherence(df_embedded, hdb7.labels_,  'HDBSCAN mcs=7')
temporal_hdb7 = temporal_coherence(df_embedded, hdb7.labels_, 'HDBSCAN mcs=7')

spatial_km    = spatial_coherence(df_embedded, km_final.labels_,  'KMeans')
temporal_km   = temporal_coherence(df_embedded, km_final.labels_, 'KMeans')


# ------------------- 6. Visualizing  -----------------------------
palette = sns.color_palette('tab20', n_colors=max(n_hdb3, n_hdb7, best_k, 1) + 1)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Perch 2 — Western Meadowlark Individual ID\nt-SNE 2D visualization',
             fontsize=14, fontweight='bold')

def plot_clusters(ax, labels, title, n_clusters):
    unique = sorted(set(labels))
    for cid in unique:
        mask  = labels == cid
        color = 'lightgrey' if cid == -1 else palette[cid % len(palette)]
        label = 'Noise' if cid == -1 else f'Ind. {cid+1} (n={mask.sum()})'
        ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
                   c=[color], s=60, alpha=0.8, label=label,
                   edgecolors='white', linewidths=0.3)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    ax.legend(fontsize=6, loc='best', markerscale=0.7,
              ncol=2 if n_clusters > 10 else 1)

# Row 1: cluster coloring
plot_clusters(axes[0, 0], hdb3.labels_,
              f'HDBSCAN mcs=3\n{n_hdb3} clusters + {n_noise_hdb3} noise', n_hdb3)
plot_clusters(axes[0, 1], hdb7.labels_,
              f'HDBSCAN mcs=7 (Sam Lapp)\n{n_hdb7} clusters + {n_noise_hdb7} noise', n_hdb7)
plot_clusters(axes[0, 2], km_final.labels_,
              f'KMeans k={best_k}\n{best_k} clusters', best_k)

# Row 2: spatial and temporal context
# Colored by time of day
time_numeric = (df_embedded['timestamp_pdt'] -
                df_embedded['timestamp_pdt'].min()).dt.total_seconds()
sc = axes[1, 0].scatter(features_2d[:, 0], features_2d[:, 1],
                         c=time_numeric, cmap='plasma', s=60, alpha=0.8,
                         edgecolors='white', linewidths=0.3)
plt.colorbar(sc, ax=axes[1, 0], label='Seconds since start')
axes[1, 0].set_title('Colored by time of day\n(do clusters = temporal territories?)',
                      fontsize=10)
axes[1, 0].set_xlabel('t-SNE dim 1')
axes[1, 0].set_ylabel('t-SNE dim 2')

# Colored by x position (easting)
sc2 = axes[1, 1].scatter(features_2d[:, 0], features_2d[:, 1],
                          c=df_embedded['x'], cmap='RdYlGn', s=60, alpha=0.8,
                          edgecolors='white', linewidths=0.3)
plt.colorbar(sc2, ax=axes[1, 1], label='Easting (m)')
axes[1, 1].set_title('Colored by X position\n(do clusters = spatial territories?)',
                      fontsize=10)
axes[1, 1].set_xlabel('t-SNE dim 1')
axes[1, 1].set_ylabel('t-SNE dim 2')

# Colored by y position (northing)
sc3 = axes[1, 2].scatter(features_2d[:, 0], features_2d[:, 1],
                          c=df_embedded['y'], cmap='RdYlBu', s=60, alpha=0.8,
                          edgecolors='white', linewidths=0.3)
plt.colorbar(sc3, ax=axes[1, 2], label='Northing (m)')
axes[1, 2].set_title('Colored by Y position\n(do clusters = spatial territories?)',
                      fontsize=10)
axes[1, 2].set_xlabel('t-SNE dim 1')
axes[1, 2].set_ylabel('t-SNE dim 2')

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'tsne_clusters.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Cluster plot saved to: {plot_path}")

# ── Temporal singing pattern plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Temporal Singing Patterns by Cluster', fontsize=13, fontweight='bold')

def plot_temporal(ax, df, labels, title, n_clusters):
    unique = [c for c in sorted(set(labels)) if c != -1]
    for cid in unique:
        mask  = labels == cid
        hours = df[mask]['timestamp_pdt'].dt.hour + \
                df[mask]['timestamp_pdt'].dt.minute / 60
        ax.scatter(hours, [cid] * mask.sum(),
                   c=[palette[cid % len(palette)]], s=30, alpha=0.6)
    ax.set_xlabel('Hour (PDT)')
    ax.set_ylabel('Cluster ID')
    ax.set_title(title, fontsize=10)
    ax.axvspan(4, 8, alpha=0.1, color='yellow', label='Dawn chorus')
    ax.legend(fontsize=7)

plot_temporal(axes[0], df_embedded, hdb3.labels_,
              f'HDBSCAN mcs=3 ({n_hdb3} clusters)', n_hdb3)
plot_temporal(axes[1], df_embedded, hdb7.labels_,
              f'HDBSCAN mcs=7 ({n_hdb7} clusters)', n_hdb7)
plot_temporal(axes[2], df_embedded, km_final.labels_,
              f'KMeans k={best_k}', best_k)

plt.tight_layout()
temporal_plot_path = os.path.join(OUTPUT_DIR, 'temporal_patterns.png')
plt.savefig(temporal_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Temporal plot saved to: {temporal_plot_path}")

# ── Spatial map coloured by cluster ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Spatial Distribution by Cluster\n(tight clusters = individual territory)',
             fontsize=13, fontweight='bold')

def plot_spatial(ax, df, labels, title):
    unique = sorted(set(labels))
    for cid in unique:
        mask  = labels == cid
        color = 'lightgrey' if cid == -1 else palette[cid % len(palette)]
        label = 'Noise' if cid == -1 else f'Ind. {cid+1}'
        ax.scatter(df[mask]['x'], df[mask]['y'],
                   c=[color], s=40, alpha=0.7, label=label,
                   edgecolors='white', linewidths=0.3)
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')
    ax.legend(fontsize=6, loc='best', markerscale=0.7,
              ncol=2 if len(set(labels)) > 10 else 1)

plot_spatial(axes[0], df_embedded, hdb3.labels_,
             f'HDBSCAN mcs=3 ({n_hdb3} clusters)')
plot_spatial(axes[1], df_embedded, hdb7.labels_,
             f'HDBSCAN mcs=7 ({n_hdb7} clusters)')
plot_spatial(axes[2], df_embedded, km_final.labels_,
             f'KMeans k={best_k}')

plt.tight_layout()
spatial_plot_path = os.path.join(OUTPUT_DIR, 'spatial_clusters.png')
plt.savefig(spatial_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Spatial plot saved to: {spatial_plot_path}")

# ------------------- 7. Export ----------------------------
df_embedded['tsne_2d_x']         = features_2d[:, 0]
df_embedded['tsne_2d_y']         = features_2d[:, 1]
df_embedded['tsne_3d_x']         = features_3d[:, 0]
df_embedded['tsne_3d_y']         = features_3d[:, 1]
df_embedded['tsne_3d_z']         = features_3d[:, 2]
df_embedded['cluster_hdbscan_3'] = hdb3.labels_
df_embedded['cluster_hdbscan_7'] = hdb7.labels_
df_embedded['cluster_kmeans']    = km_final.labels_

results_path = os.path.join(OUTPUT_DIR, 'individual_id_results.csv')
df_embedded.to_csv(results_path, index=False)
print(f"  Results saved to: {results_path}")

# Save coherence tables
spatial_hdb3.to_csv(os.path.join(OUTPUT_DIR, 'spatial_coherence_hdbscan3.csv'), index=False)
spatial_hdb7.to_csv(os.path.join(OUTPUT_DIR, 'spatial_coherence_hdbscan7.csv'), index=False)
spatial_km.to_csv(os.path.join(OUTPUT_DIR,   'spatial_coherence_kmeans.csv'),   index=False)

# ---------------- 8. Summary -----------------------------
print(f"  Total detections processed:           {len(df_embedded)}")
print(f"  HDBSCAN (mcs=3) clusters:             {n_hdb3}  (+ {n_noise_hdb3} unassigned)")
print(f"  HDBSCAN (mcs=7) clusters:             {n_hdb7}  (+ {n_noise_hdb7} unassigned)")
print(f"  KMeans clusters (best k):             {best_k}")
print(f"\n  Use spatial_clusters.png to validate:")
print(f"  → Clusters that are spatially tight = likely one individual's territory")
print(f"  → Clusters scattered across the grid = likely noise or merged individuals")
print(f"\n  Use temporal_patterns.png to validate:")
print(f"  → Each cluster should show realistic dawn chorus timing (04:00-08:00 PDT)")
print(f"  → Clusters active all day or at odd hours may be artifacts")